# SQL Analysis: FIFA World Cup Results & Scouting Fundamentals

**SQL portfolio companion** | DuckDB · Python · pandas · Google Colab

## Project overview

This notebook demonstrates core and intermediate SQL techniques using a **fixed snapshot of international football results** and a **small, explicitly fictional player-scouting example**. It complements the separate [World Cup Python analysis](01_world_cup_exploration.ipynb).

**Questions explored**

- How many FIFA World Cup matches and goals are recorded in this dataset?
- How do scoring averages vary by tournament year?
- Which tournaments meet specific thresholds for match volume or scoring rate?
- How do `INNER JOIN` and per-90 normalization change a simple scouting comparison?
- How did tournament scoring averages change relative to the previous World Cup?

**Source:** [International Football Results, martj42/international_results](https://github.com/martj42/international_results). The CSV URL in the setup cell points to a **specific Git commit** for reproducibility. All findings refer to this dataset snapshot, not a live feed.

**How to run:** Open in Google Colab and select **Runtime → Run all**. The first code cell installs DuckDB, loads the CSV, and registers a SQL table named `matches`. An internet connection is needed to install/load dependencies and download the dataset. Subsequent cells depend on the setup cell; they can fail if the Colab runtime resets.

> **Scope note:** This is a technical SQL demonstration and exploratory study, not an estimate of causal impacts or a complete player-recruitment model.

## 1. SQL Basics — Filtering and Sorting

Load the data into DuckDB, then use `SELECT`, `WHERE`, `ORDER BY`, and `LIMIT` to inspect the earliest FIFA World Cup records. Match date is only a calendar date; matches played on the same day cannot be ordered by kickoff time from this field alone.

In [15]:

!pip install -q duckdb

import pandas as pd
import duckdb

# Load our World Cup project dataset
url = (
    "https://raw.githubusercontent.com/"
    "martj42/international_results/"
    "b7a3a8ee5d37780c6a03101c188658fd439e241d/"
    "results.csv"
)

df = pd.read_csv(url)

# Create a SQL connection
con = duckdb.connect()

# Make the DataFrame available as a SQL table
con.register("matches", df)

print("SQL database ready!")


SQL database ready!


In [16]:


result = con.execute("""
    SELECT
        date,
        home_team,
        away_team,
        home_score,
        away_score
    FROM matches
    WHERE tournament = 'FIFA World Cup'
    ORDER BY date ASC
    LIMIT 10
""").df()

display(result)



,date,home_team,away_team,home_score,away_score
0,1930-07-13,Belgium,United States,0,3
1,1930-07-13,France,Mexico,4,1
2,1930-07-14,Peru,Romania,1,3
3,1930-07-14,Brazil,Yugoslavia,1,2
4,1930-07-15,Argentina,France,1,0
5,1930-07-16,Chile,Mexico,3,0
6,1930-07-17,Paraguay,United States,0,3
7,1930-07-17,Bolivia,Yugoslavia,0,4
8,1930-07-18,Uruguay,Peru,1,0
9,1930-07-19,Argentina,Mexico,6,3


**Interpretation:** The first displayed records start in July 1930. This query shows individual matches; it does not calculate any tournament-level statistics. The leftmost `0, 1, …` is the pandas display index, not a match identifier.

## 2. SQL Aggregations

`COUNT(*)` counts rows, `SUM()` adds numeric values, and `AVG()` calculates the mean. Filtering with `WHERE tournament = 'FIFA World Cup'` ensures the aggregates refer only to World Cup fixtures in this source.

In [17]:

result = con.execute("""
    SELECT
        tournament,
        COUNT(*) AS total_matches
    FROM matches
    GROUP BY tournament
    ORDER BY total_matches DESC
    LIMIT 10
""").df()

display(result)


,tournament,total_matches
0,Friendly,18263
1,FIFA World Cup qualification,8771
2,UEFA Euro qualification,2824
3,African Cup of Nations qualification,2387
4,FIFA World Cup,1068
5,Copa América,869
6,African Cup of Nations,845
7,AFC Asian Cup qualification,829
8,UEFA Nations League,762
9,CECAFA Cup,620


In [18]:

result = con.execute("""
    SELECT
        COUNT(*) AS world_cup_matches
    FROM matches
    WHERE tournament = 'FIFA World Cup'
""").df()

print(result.to_string(index=False))

 world_cup_matches
              1068


In [19]:

result = con.execute("""
    SELECT
        SUM(home_score + away_score) AS total_goals
    FROM matches
    WHERE tournament = 'FIFA World Cup'
""").df()

print(result.to_string(index=False))

 total_goals
      3028.0


In [20]:
result = con.execute("""
    SELECT
        ROUND(AVG(home_score + away_score),2) AS avg_goals
    FROM matches
    WHERE tournament = 'FIFA World Cup'
""").df()

print(result.to_string(index=False))

 avg_goals
      2.84


**Finding:** The pinned dataset contains **1,068 World Cup match records** and **3,028 combined goals**, equivalent to approximately **2.84 goals per match** across the included tournaments. These figures describe recorded matches in the specified data snapshot.

## 3. GROUP BY — World Cup Analysis

`GROUP BY` creates one aggregate row per year. `YEAR(CAST(date AS DATE))` extracts the year from the match-date field, and `ROUND(AVG(...), 2)` displays each tournament's goals-per-match average.

In [21]:

result = con.execute("""
    SELECT
        YEAR(CAST(date AS DATE)) AS year,
        COUNT(*) AS total_matches,
        ROUND(AVG(home_score + away_score),2) AS avg_goals
    FROM matches
    WHERE tournament = 'FIFA World Cup'
    GROUP BY year
    ORDER BY year ASC
""").df()

display(result)


,year,total_matches,avg_goals
0,1930,18,3.89
1,1934,17,4.12
2,1938,18,4.67
3,1950,22,4.00
4,1954,26,5.38
5,1958,35,3.60
6,1962,32,2.78
7,1966,32,2.78
8,1970,32,2.97
9,1974,38,2.55


**Finding:** In this dataset, **1954 (5.38)** has the highest tournament scoring average and **1990 (2.21)** the lowest. Match counts differ markedly between tournaments, so average goals per match is more comparable across formats than raw goal totals alone.

## 4. Filtering Aggregated Data with HAVING

`WHERE` filters match records **before** aggregation. `HAVING` filters the resulting **groups**, such as years with at least 60 matches or years averaging more than three goals per match.

In [22]:

result = con.execute("""
    SELECT
        YEAR(CAST(date AS DATE)) AS year,
        COUNT(*) AS total_matches,
        ROUND(AVG(home_score + away_score), 2) AS avg_goals
    FROM matches
    WHERE tournament = 'FIFA World Cup'
    GROUP BY year
    HAVING COUNT(*) >= 60
    ORDER BY year ASC
""").df()

display(result)


,year,total_matches,avg_goals
0,1998,64,2.67
1,2002,64,2.52
2,2006,64,2.30
3,2010,64,2.27
4,2014,64,2.67
5,2018,64,2.64
6,2022,64,2.69
7,2026,104,2.96


In [23]:
result = con.execute("""
    SELECT
        YEAR(CAST(date AS DATE)) AS year,
        COUNT(*) AS total_matches,
        ROUND(AVG(home_score + away_score), 2) AS avg_goals
    FROM matches
    WHERE tournament = 'FIFA World Cup'
    GROUP BY year
    HAVING AVG(home_score + away_score) > 3
    ORDER BY year ASC
""").df()

display(result)

,year,total_matches,avg_goals
0,1930,18,3.89
1,1934,17,4.12
2,1938,18,4.67
3,1950,22,4.00
4,1954,26,5.38
5,1958,35,3.60


**Finding:** Six tournaments in this dataset averaged more than **3.00 goals per match**, and all occurred between **1930 and 1958**. Historical tactical structures could be part of an explanation, but this match-results dataset does **not** include formations, possession, or chance-creation data; it cannot establish the cause of the difference.

## 5. SQL JOINs — Player Scouting Analysis

### Example: Joining player information to performance data

The following three-player dataset is **fictional and used for SQL practice only**. It is not sourced from an actual club, season, or scouting platform. `player_id` links the player details and performance tables and avoids the ambiguity of matching by name.

In [24]:

# Fictional player scouting dataset

players = pd.DataFrame({
    "player_id": [101, 102, 103],
    "player_name": ["Player A", "Player B", "Player C"],
    "position": ["CM", "CM", "CM"]
})

performance = pd.DataFrame({
    "player_id": [101, 102, 103],
    "minutes_played": [1800, 1200, 2200],
    "key_passes": [45, 40, 55]
})

# Register both tables in DuckDB
con.register("players", players)
con.register("performance", performance)

print("PLAYERS TABLE")
display(players)

print("PERFORMANCE TABLE")
display(performance)


PLAYERS TABLE


,player_id,player_name,position
0,101,Player A,CM
1,102,Player B,CM
2,103,Player C,CM


PERFORMANCE TABLE


,player_id,minutes_played,key_passes
0,101,1800,45
1,102,1200,40
2,103,2200,55


In [25]:

result = con.execute("""
    SELECT
        p.player_name,
        p.position,
        s.minutes_played,
        s.key_passes
    FROM players AS p
    INNER JOIN performance AS s
        ON p.player_id = s.player_id
""").df()

display(result)


,player_name,position,minutes_played,key_passes
0,Player A,CM,1800,45
1,Player B,CM,1200,40
2,Player C,CM,2200,55


### Comparing players fairly: per-90 normalization

Counting total key passes alone favors players with more playing time. `key_passes * 90 / minutes_played` converts totals into a common per-90-minute rate. `NULLIF(minutes_played, 0)` avoids a divide-by-zero error for players with no minutes.

In [26]:

result = con.execute("""
    SELECT
        p.player_name,
        p.position,
        s.minutes_played,
        s.key_passes,
        ROUND(
            s.key_passes * 90.0 /
            NULLIF(s.minutes_played, 0),
            2
        ) AS key_passes_per_90
    FROM players AS p
    INNER JOIN performance AS s
        ON p.player_id = s.player_id
    ORDER BY key_passes_per_90 DESC,
             p.player_name ASC
""").df()

display(result)


,player_name,position,minutes_played,key_passes,key_passes_per_90
0,Player B,CM,1200,40,3.00
1,Player A,CM,1800,45,2.25
2,Player C,CM,2200,55,2.25


**Finding (fictional example):** Player C has the most total key passes (**55**), while Player B has the highest rate (**3.00 per 90 minutes**), ahead of Players A and C (**2.25** each). A real scouting recommendation would also require league strength, role, sample size, minutes threshold, other metrics, video review, and recruitment constraints; **key passes per 90 are not a stand-alone recruitment decision**.

## 6. Advanced SQL — CTEs and Window Functions

A **common table expression (CTE)**, written with `WITH`, calculates yearly statistics for the next query. The **`LAG()` window function** references the preceding tournament to quantify the change in scoring average. The comparison is with the **previous World Cup**, not necessarily the previous calendar year. The displayed change uses averages already rounded to two decimal places.

In [27]:

result = con.execute("""
    WITH yearly_stats AS (
        SELECT
            YEAR(CAST(date AS DATE)) AS year,
            COUNT(*) AS total_matches,
            ROUND(
                AVG(home_score + away_score), 2
            ) AS avg_goals
        FROM matches
        WHERE tournament = 'FIFA World Cup'
        GROUP BY year
    )

    SELECT
        year,
        total_matches,
        avg_goals,
        LAG(avg_goals) OVER (
            ORDER BY year
        ) AS previous_world_cup_avg,
        ROUND(
            avg_goals -
            LAG(avg_goals) OVER (ORDER BY year),
            2
        ) AS change_from_previous
    FROM yearly_stats
    ORDER BY year
""").df()

display(result)


,year,total_matches,avg_goals,previous_world_cup_avg,change_from_previous
0,1930,18,3.89,NaN,NaN
1,1934,17,4.12,3.89,0.23
2,1938,18,4.67,4.12,0.55
3,1950,22,4.00,4.67,-0.67
4,1954,26,5.38,4.00,1.38
5,1958,35,3.60,5.38,-1.78
6,1962,32,2.78,3.60,-0.82
7,1966,32,2.78,2.78,0.00
8,1970,32,2.97,2.78,0.19
9,1974,38,2.55,2.97,-0.42


## Conclusions and analytical considerations

- **Historical scoring:** World Cup goals-per-match averages vary substantially by tournament in the pinned dataset; **1954** is the high point at **5.38**, while **1990** is the low point at **2.21**.
- **Recent comparison:** In this snapshot, the average rises from **2.69 in 2022** to **2.96 in 2026**, a displayed change of **+0.27 goals per match**. Tournament size and competitive context also changed, so the change should **not** be attributed to expansion without additional evidence.
- **Good analytical practice:** Normalize counts when exposure differs (such as goals per match or key passes per 90); use `HAVING` for aggregate filters; use IDs to join related tables; and distinguish observed patterns from explanations.

### Limitations

1. The CSV is a fixed historical snapshot with tournament labels and scorelines, not detailed event, tactical, player, opponent-strength, or commercial data.
2. Tournament formats and match counts vary across years; simple comparisons do not control for these differences.
3. The three-player JOIN example is synthetic and exists solely to demonstrate a relational query and per-90 normalization.
4. Differences in displayed averages may reflect two-decimal rounding.

**Portfolio context:** This SQL notebook demonstrates querying and reproducible analytical reasoning. More decision-oriented sports recruitment and business-analytics portfolio projects will build on these skills with richer, appropriately sourced datasets.